In [ ]:
"""
Darija TTS Dataset Builder with WhisperX Forced Alignment (Colab GPU)

This script is organized in Colab-style cells:
- Copy/paste each `## Cell` block into Colab, or run as one script in Colab.
- No transcript rewriting: original Darija text is preserved.
- Whisper ASR text is used only for rough timestamp anchors.
"""

# =============================================================================


In [ ]:
# ## Cell 1: Title + Usage Notes (Markdown in notebook)
# =============================================================================
# Darija TTS Dataset Builder with WhisperX Forced Alignment (Colab GPU)
#
# Critical guarantees:
# - No transcript rewriting: original Darija text is preserved.
# - Whisper ASR text is never used as final transcript.
# - Final metadata text comes from your provided transcript segments.


# =============================================================================


In [ ]:
# ## Cell 2: User-configurable parameters
# =============================================================================
from dataclasses import dataclass


@dataclass
class Config:
    zip_path: str = "/mnt/data/cafe-small-clean.zip"
    extract_root: str = "/content/data"
    output_dir: str = "/content/tts_export"
    target_sr: int = 16000

    min_dur: float = 1.5
    max_dur: float = 15.0
    preferred_max_dur: float = 12.0

    pause_gap_threshold: float = 0.6
    coverage_threshold: float = 0.70

    silence_trim_ms: int = 200
    padding_ms: int = 80

    whisper_model_size: str = "medium"
    batch_size: int = 8
    compute_type: str = "float16"

    seed: int = 42
    run_optional_hf_cell: bool = False


CONFIG = Config()
print(CONFIG)


# =============================================================================


In [ ]:
# ## Cell 3: Environment checks + installs
# =============================================================================
import importlib.metadata as importlib_metadata
import os
import subprocess
import sys


def run(cmd: str, check: bool = True):
    print(f"+ {cmd}")
    return subprocess.run(cmd, shell=True, check=check)


def run_capture(cmd: str):
    print(f"+ {cmd}")
    return subprocess.run(cmd, shell=True, check=False, text=True, capture_output=True)


gpu_check = run_capture("nvidia-smi -L")
if gpu_check.returncode != 0 or "GPU" not in gpu_check.stdout:
    raise SystemExit(
        "NVIDIA GPU runtime is required. In Colab: Runtime -> Change runtime type -> GPU."
    )
print(gpu_check.stdout.strip())

run("apt-get -y update")
run("apt-get -y install ffmpeg")
run(f"{sys.executable} -m pip install -q --upgrade pip")
run(
    f"{sys.executable} -m pip install -q pandas soundfile librosa tqdm datasets ipywidgets"
)

primary = run_capture(f"{sys.executable} -m pip install -q whisperx==3.7.6")
if primary.returncode != 0:
    print("Primary whisperx==3.7.6 install failed. Applying fallback pins...")
    print(primary.stderr[-4000:])
    run(
        f"{sys.executable} -m pip install -q "
        "'ctranslate2>=4.5.0,<5' 'faster-whisper>=1.1.1,<1.3'"
    )
    run(f"{sys.executable} -m pip install -q whisperx==3.7.2")

import torch  # noqa: E402
import torchaudio  # noqa: E402
import whisperx  # noqa: E402

print("\nInstalled versions:")
print("torch:", torch.__version__)
print("torchaudio:", torchaudio.__version__)
print("whisperx:", whisperx.__version__ if hasattr(whisperx, "__version__") else "unknown")
try:
    print("ctranslate2:", importlib_metadata.version("ctranslate2"))
except Exception:
    print("ctranslate2: not found")
ffmpeg_v = run_capture("ffmpeg -version")
print(ffmpeg_v.stdout.splitlines()[0] if ffmpeg_v.stdout else "ffmpeg not found")


# =============================================================================


In [ ]:
# ## Cell 4: Imports + utility setup
# =============================================================================
import csv
import json
import random
import re
import shutil
import traceback
import zipfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

random.seed(CONFIG.seed)
np.random.seed(CONFIG.seed)

OUTPUT_DIR = Path(CONFIG.output_dir)
WAVS_DIR = OUTPUT_DIR / "wavs"
INTERMEDIATE_DIR = OUTPUT_DIR / "intermediate"
LOGS_DIR = OUTPUT_DIR / "logs"
ALIGNED_WORDS_DIR = INTERMEDIATE_DIR / "aligned_words"
TEMP_DIR = INTERMEDIATE_DIR / "temp_audio"

for p in [OUTPUT_DIR, WAVS_DIR, INTERMEDIATE_DIR, LOGS_DIR, ALIGNED_WORDS_DIR, TEMP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Output directories ready:")
print("-", OUTPUT_DIR)
print("-", WAVS_DIR)
print("-", INTERMEDIATE_DIR)
print("-", LOGS_DIR)


# =============================================================================


In [ ]:
# ## Cell 5: Unzip + transcript discovery
# =============================================================================
ZIP_PATH = Path(CONFIG.zip_path)
EXTRACT_ROOT = Path(CONFIG.extract_root)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Zip file not found: {ZIP_PATH}")

print(f"Unzipping {ZIP_PATH} -> {EXTRACT_ROOT}")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)


def find_transcript_csv(root: Path) -> Path:
    candidates = sorted(root.rglob("*.csv"))
    valid = []
    for p in candidates:
        try:
            head = pd.read_csv(p, nrows=5)
        except Exception:
            continue
        cols = {str(c).strip().lower() for c in head.columns}
        if {"path", "text"}.issubset(cols):
            valid.append(p)
    if not valid:
        raise FileNotFoundError("No transcript CSV found with required columns: path,text")
    return valid[0]


transcript_csv_path = find_transcript_csv(EXTRACT_ROOT)
print("Transcript CSV selected:", transcript_csv_path)

transcripts_df = pd.read_csv(transcript_csv_path)
transcripts_df.columns = [str(c).strip().lower() for c in transcripts_df.columns]
transcripts_df = transcripts_df[["path", "text"]].copy()
transcripts_df["path"] = transcripts_df["path"].fillna("").astype(str).str.strip()
transcripts_df["text"] = transcripts_df["text"].fillna("").astype(str)
transcripts_df = transcripts_df[
    ~((transcripts_df["path"] == "") & (transcripts_df["text"].str.strip() == ""))
].copy()
transcripts_df = transcripts_df.reset_index(drop=True)
transcripts_df["row_id"] = transcripts_df.index.astype(int)

print("Total transcript rows:", len(transcripts_df))
print(transcripts_df.head(3))


# =============================================================================


In [ ]:
# ## Cell 6: Robust path resolution + report
# =============================================================================
def norm_rel_path(s: str) -> str:
    s = s.replace("\\", "/").strip()
    s = re.sub(r"^\./+", "", s)
    s = re.sub(r"/+", "/", s)
    return s.lower()


all_wavs = sorted(EXTRACT_ROOT.rglob("*.wav"))
print("Discovered WAV files:", len(all_wavs))

rel_index = {}
basename_index = {}
for wav in all_wavs:
    rel = norm_rel_path(str(wav.relative_to(EXTRACT_ROOT)))
    rel_index.setdefault(rel, []).append(wav)
    basename_index.setdefault(wav.name.lower(), []).append(wav)


def resolve_audio_path(raw_path: str):
    raw = raw_path.strip().replace("\\", "/")
    raw = re.sub(r"^\./+", "", raw)

    candidates = [(raw, "direct_norm")]
    if raw.lower().startswith("cafe/"):
        candidates.append((raw[5:], "strip_cafe"))
    prefix = "cafe/cafe-small-clean/"
    if raw.lower().startswith(prefix):
        candidates.append((raw[len(prefix):], "strip_cafe_cafe-small-clean"))

    for cand, method in candidates:
        k = norm_rel_path(cand)
        hits = rel_index.get(k, [])
        if len(hits) == 1:
            return str(hits[0]), method
        if len(hits) > 1:
            return str(hits[0]), method + "_ambiguous_pick_first"

    base = Path(raw).name.lower()
    base_hits = basename_index.get(base, [])
    if len(base_hits) == 1:
        return str(base_hits[0]), "basename_unique"
    if len(base_hits) > 1:
        return None, "basename_ambiguous"

    return None, "not_found"


resolved_paths = []
resolve_methods = []
for p in transcripts_df["path"].tolist():
    rp, rm = resolve_audio_path(p)
    resolved_paths.append(rp)
    resolve_methods.append(rm)

transcripts_df["resolved_audio_path"] = resolved_paths
transcripts_df["resolve_method"] = resolve_methods
transcripts_df["audio_found"] = transcripts_df["resolved_audio_path"].notna()

found_count = int(transcripts_df["audio_found"].sum())
missing_count = int((~transcripts_df["audio_found"]).sum())
print(f"Rows total: {len(transcripts_df)}")
print(f"Found audio: {found_count}")
print(f"Missing audio: {missing_count}")

path_report = transcripts_df[
    ["row_id", "path", "text", "resolved_audio_path", "resolve_method", "audio_found"]
].copy()
path_report.to_csv(LOGS_DIR / "path_resolution_report.csv", index=False)
print("Saved:", LOGS_DIR / "path_resolution_report.csv")
print(path_report.head(5))


# =============================================================================


In [ ]:
# ## Cell 7: Pre-alignment audio QC
# =============================================================================
qc_records = []
for row in tqdm(transcripts_df.itertuples(index=False), total=len(transcripts_df), desc="Pre-QC"):
    if not row.audio_found:
        continue
    audio_path = Path(row.resolved_audio_path)
    try:
        y, sr = sf.read(audio_path, always_2d=True)
        n_samples, n_channels = y.shape
        peak = float(np.max(np.abs(y))) if y.size else 0.0
        duration = float(n_samples / sr) if sr > 0 else 0.0
        qc_records.append(
            {
                "row_id": int(row.row_id),
                "audio_path": str(audio_path),
                "duration_sec": duration,
                "sample_rate": int(sr),
                "channels": int(n_channels),
                "peak_amplitude": peak,
                "is_clipping": bool(peak >= 0.999),
            }
        )
    except Exception as e:
        qc_records.append(
            {
                "row_id": int(row.row_id),
                "audio_path": str(audio_path),
                "duration_sec": np.nan,
                "sample_rate": np.nan,
                "channels": np.nan,
                "peak_amplitude": np.nan,
                "is_clipping": np.nan,
                "qc_error": str(e),
            }
        )

qc_pre_df = pd.DataFrame(qc_records)
qc_pre_path = OUTPUT_DIR / "qc_pre_alignment.csv"
qc_pre_df.to_csv(qc_pre_path, index=False)
ok_qc = qc_pre_df.dropna(subset=["duration_sec"]).copy()

if len(ok_qc) > 0:
    summary = {
        "count": len(ok_qc),
        "min_dur": float(ok_qc["duration_sec"].min()),
        "mean_dur": float(ok_qc["duration_sec"].mean()),
        "median_dur": float(ok_qc["duration_sec"].median()),
        "max_dur": float(ok_qc["duration_sec"].max()),
        "pct_gt_15s": float((ok_qc["duration_sec"] > 15).mean() * 100),
        "pct_gt_60s": float((ok_qc["duration_sec"] > 60).mean() * 100),
        "pct_clipping": float((ok_qc["peak_amplitude"] >= 0.999).mean() * 100),
    }
    print("Pre-alignment QC summary:")
    for k, v in summary.items():
        print(f"- {k}: {v:.4f}" if isinstance(v, float) else f"- {k}: {v}")
else:
    print("No valid QC rows available.")

print("Saved:", qc_pre_path)
print(qc_pre_df.head(5))


# =============================================================================


In [ ]:
# ## Cell 8: Load WhisperX models + helper functions
# =============================================================================
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook. Enable GPU runtime in Colab.")

device = "cuda"
print("Loading ASR model for rough timestamps only...")
asr_model = whisperx.load_model(
    CONFIG.whisper_model_size,
    device=device,
    compute_type=CONFIG.compute_type,
)

print("Loading alignment model for Arabic...")
align_model, align_metadata = whisperx.load_align_model(language_code="ar", device=device)

WORD_RE = re.compile(r"[A-Za-z0-9\u0600-\u06FF']+", re.UNICODE)
PUNCT_CHARS = set(".!?؟،؛…")


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def word_tokenize_for_coverage(text: str):
    return WORD_RE.findall(str(text))


def split_text_by_punctuation(text: str):
    text = str(text)
    chunks = []
    current = []
    for ch in text:
        current.append(ch)
        if ch in PUNCT_CHARS:
            c = "".join(current).strip()
            if c:
                chunks.append(c)
            current = []
    tail = "".join(current).strip()
    if tail:
        chunks.append(tail)
    if not chunks and text.strip():
        chunks = [text.strip()]
    return chunks


def build_rough_segments(audio_path: str):
    audio = whisperx.load_audio(audio_path)
    asr_result = asr_model.transcribe(audio, batch_size=CONFIG.batch_size, language="ar")
    rough = []
    for seg in asr_result.get("segments", []):
        try:
            s = float(seg.get("start", 0.0))
            e = float(seg.get("end", 0.0))
        except Exception:
            continue
        if e <= s:
            continue
        rough.append({"start": s, "end": e, "text": str(seg.get("text", "")).strip()})
    return rough


def make_fixed_windows(audio_duration: float, window_sec: float = 20.0):
    windows = []
    t = 0.0
    while t < audio_duration:
        e = min(audio_duration, t + window_sec)
        windows.append({"start": t, "end": e, "text": ""})
        t = e
    if not windows and audio_duration > 0:
        windows = [{"start": 0.0, "end": audio_duration, "text": ""}]
    return windows


def distribute_transcript_over_segments(transcript: str, rough_segments: list, audio_duration: float):
    words = transcript.split()
    if not words:
        return []

    segments = list(rough_segments)
    if not segments:
        segments = make_fixed_windows(audio_duration=audio_duration, window_sec=20.0)

    durations = [max(0.01, float(seg["end"] - seg["start"])) for seg in segments]
    total_dur = float(sum(durations))

    alloc = [max(1, int(round(len(words) * (d / total_dur)))) for d in durations]
    while sum(alloc) > len(words):
        idx = int(np.argmax(alloc))
        if alloc[idx] > 1:
            alloc[idx] -= 1
        else:
            break
    while sum(alloc) < len(words):
        idx = int(np.argmax(durations))
        alloc[idx] += 1

    forced_segments = []
    cursor = 0
    for seg, n_words in zip(segments, alloc):
        chunk_words = words[cursor : cursor + n_words]
        cursor += n_words
        chunk_text = " ".join(chunk_words).strip()
        if not chunk_text:
            continue
        forced_segments.append(
            {
                "start": float(seg["start"]),
                "end": float(seg["end"]),
                "text": chunk_text,
            }
        )

    if cursor < len(words):
        leftover = " ".join(words[cursor:]).strip()
        if forced_segments:
            forced_segments[-1]["text"] = (forced_segments[-1]["text"] + " " + leftover).strip()
        else:
            forced_segments.append({"start": 0.0, "end": float(audio_duration), "text": leftover})

    if not forced_segments:
        forced_segments = [{"start": 0.0, "end": float(audio_duration), "text": transcript.strip()}]

    return forced_segments


print("WhisperX models loaded and helper functions ready.")


# =============================================================================


In [ ]:
# ## Cell 9: Forced alignment loop (preserve provided transcript)
# =============================================================================
qc_duration_map = {
    int(r.row_id): float(r.duration_sec)
    for r in qc_pre_df.dropna(subset=["duration_sec"]).itertuples(index=False)
}

alignment_rows = []
processing_errors = []
aligned_ok_cache = {}

rows_for_alignment = transcripts_df[transcripts_df["audio_found"]].copy()
for row in tqdm(rows_for_alignment.itertuples(index=False), total=len(rows_for_alignment), desc="Forced alignment"):
    row_id = int(row.row_id)
    raw_text = str(row.text)
    audio_path = str(row.resolved_audio_path)

    try:
        duration = float(qc_duration_map.get(row_id, 0.0))
        if duration <= 0:
            info = sf.info(audio_path)
            duration = float(info.frames / info.samplerate)

        if normalize_whitespace(raw_text) == "":
            alignment_rows.append(
                {
                    "row_id": row_id,
                    "audio_path": audio_path,
                    "total_words": 0,
                    "aligned_words": 0,
                    "coverage": 0.0,
                    "status": "needs_review",
                    "reason": "empty_text",
                }
            )
            continue

        rough_segments = build_rough_segments(audio_path)
        forced_segments = distribute_transcript_over_segments(raw_text, rough_segments, audio_duration=duration)

        audio_arr = whisperx.load_audio(audio_path)
        aligned_result = whisperx.align(
            forced_segments,
            align_model,
            align_metadata,
            audio_arr,
            device,
            return_char_alignments=False,
        )

        word_segments = aligned_result.get("word_segments")
        if word_segments is None:
            word_segments = []
            for s in aligned_result.get("segments", []):
                word_segments.extend(s.get("words", []))

        valid_words = []
        for w in word_segments:
            try:
                ws = float(w.get("start"))
                we = float(w.get("end"))
            except Exception:
                continue
            if not np.isfinite(ws) or not np.isfinite(we) or we <= ws:
                continue
            token = str(w.get("word", "")).strip()
            valid_words.append({"start": ws, "end": we, "word": token})

        orig_words = raw_text.split()
        for i, w in enumerate(valid_words):
            w["orig_word"] = orig_words[i] if i < len(orig_words) else w.get("word", "")

        total_words = len(word_tokenize_for_coverage(raw_text))
        aligned_words = len(valid_words)
        coverage = float(aligned_words / max(1, total_words))
        status = "ok" if coverage >= CONFIG.coverage_threshold else "needs_review"

        aligned_json_path = ALIGNED_WORDS_DIR / f"{row_id:06d}.json"
        with open(aligned_json_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "row_id": row_id,
                    "audio_path": audio_path,
                    "duration": duration,
                    "raw_text": raw_text,
                    "coverage": coverage,
                    "status": status,
                    "forced_segments": forced_segments,
                    "aligned_words": valid_words,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        alignment_rows.append(
            {
                "row_id": row_id,
                "audio_path": audio_path,
                "total_words": total_words,
                "aligned_words": aligned_words,
                "coverage": coverage,
                "status": status,
                "reason": "",
                "aligned_json": str(aligned_json_path),
            }
        )

        if status == "ok":
            aligned_ok_cache[row_id] = {
                "row_id": row_id,
                "audio_path": audio_path,
                "duration": duration,
                "raw_text": raw_text,
                "aligned_words": valid_words,
            }
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        alignment_rows.append(
            {
                "row_id": row_id,
                "audio_path": audio_path,
                "total_words": np.nan,
                "aligned_words": np.nan,
                "coverage": 0.0,
                "status": "error",
                "reason": msg,
            }
        )
        processing_errors.append(
            {
                "row_id": row_id,
                "audio_path": audio_path,
                "stage": "alignment",
                "error": msg,
                "traceback": traceback.format_exc(),
            }
        )

alignment_report_df = pd.DataFrame(alignment_rows)
alignment_report_path = OUTPUT_DIR / "alignment_report.csv"
alignment_report_df.to_csv(alignment_report_path, index=False)

print("Alignment done.")
print("- Total attempted:", len(rows_for_alignment))
print("- OK:", int((alignment_report_df["status"] == "ok").sum()))
print("- Needs review:", int((alignment_report_df["status"] == "needs_review").sum()))
print("- Errors:", int((alignment_report_df["status"] == "error").sum()))
print("Saved:", alignment_report_path)
print(alignment_report_df.head(5))


# =============================================================================


In [ ]:
# ## Cell 10: Segmentation (punctuation + pauses + duration shaping)
# =============================================================================
def words_duration(words):
    if not words:
        return 0.0
    return float(words[-1]["end"] - words[0]["start"])


def text_from_words(words):
    toks = []
    for w in words:
        t = str(w.get("orig_word") or w.get("word") or "").strip()
        if t:
            toks.append(t)
    txt = " ".join(toks)
    txt = re.sub(r"\s+([\.\!\?؟،؛…])", r"\1", txt)
    return normalize_whitespace(txt)


def sentence_word_counts(raw_text: str):
    sents = split_text_by_punctuation(raw_text)
    counts = [max(1, len(word_tokenize_for_coverage(s))) for s in sents]
    return sents, counts


def allocate_words_to_sentences(aligned_words, raw_text):
    sents, counts = sentence_word_counts(raw_text)
    if not aligned_words:
        return []

    chunks = []
    cursor = 0
    n = len(aligned_words)
    for i, (sent_text, cnt) in enumerate(zip(sents, counts)):
        if i == len(sents) - 1:
            part = aligned_words[cursor:]
        else:
            part = aligned_words[cursor : min(n, cursor + cnt)]
        cursor += cnt
        if part:
            chunks.append({"text": normalize_whitespace(sent_text), "words": part})

    if not chunks:
        chunks = [{"text": text_from_words(aligned_words), "words": aligned_words}]

    assigned = sum(len(c["words"]) for c in chunks)
    if assigned < n:
        leftovers = aligned_words[assigned:]
        if chunks:
            chunks[-1]["words"].extend(leftovers)
        else:
            chunks.append({"text": text_from_words(leftovers), "words": leftovers})
    return chunks


def split_words_by_pause(words, gap_threshold):
    if not words:
        return []
    parts = []
    start_idx = 0
    for i in range(len(words) - 1):
        gap = float(words[i + 1]["start"] - words[i]["end"])
        if gap > gap_threshold:
            parts.append(words[start_idx : i + 1])
            start_idx = i + 1
    parts.append(words[start_idx:])
    return [p for p in parts if p]


def split_chunk_to_target(words, target_max_dur):
    if not words:
        return []
    dur = words_duration(words)
    if dur <= target_max_dur or len(words) <= 1:
        return [words]

    candidates = []
    for i in range(len(words) - 1):
        gap = float(words[i + 1]["start"] - words[i]["end"])
        left_dur = float(words[i]["end"] - words[0]["start"])
        right_dur = float(words[-1]["end"] - words[i + 1]["start"])
        balance = abs(left_dur - right_dur)
        candidates.append((i, gap, balance))

    if candidates:
        candidates = sorted(candidates, key=lambda x: (-x[1], x[2]))
        split_idx = candidates[0][0] + 1
    else:
        split_idx = len(words) // 2

    left = words[:split_idx]
    right = words[split_idx:]
    out = []
    out.extend(split_chunk_to_target(left, target_max_dur))
    out.extend(split_chunk_to_target(right, target_max_dur))
    return out


def make_segment(words, default_text=None):
    return {
        "start": float(words[0]["start"]),
        "end": float(words[-1]["end"]),
        "duration": float(words[-1]["end"] - words[0]["start"]),
        "words": words,
        "text": normalize_whitespace(default_text if default_text else text_from_words(words)),
    }


def combine_segments(seg_a, seg_b):
    words = seg_a["words"] + seg_b["words"]
    words = sorted(words, key=lambda x: (x["start"], x["end"]))
    return make_segment(words)


segment_candidates = []
dropped_segments = []

for row_id, rec in tqdm(aligned_ok_cache.items(), total=len(aligned_ok_cache), desc="Segmentation"):
    aligned_words = rec["aligned_words"]
    raw_text = rec["raw_text"]
    if not aligned_words:
        dropped_segments.append(
            {
                "row_id": row_id,
                "segment_id": "",
                "reason": "no_aligned_words",
                "duration": 0.0,
                "text": raw_text,
            }
        )
        continue

    sentence_chunks = allocate_words_to_sentences(aligned_words, raw_text)
    pause_chunks = []
    for c in sentence_chunks:
        split_parts = split_words_by_pause(c["words"], CONFIG.pause_gap_threshold)
        if len(split_parts) == 1:
            pause_chunks.append(make_segment(split_parts[0], default_text=c["text"]))
        else:
            for p in split_parts:
                pause_chunks.append(make_segment(p))

    shaped = []
    for seg in pause_chunks:
        if seg["duration"] > CONFIG.preferred_max_dur:
            for sp in split_chunk_to_target(seg["words"], CONFIG.preferred_max_dur):
                shaped.append(make_segment(sp))
        else:
            shaped.append(seg)

    shaped = sorted(shaped, key=lambda x: (x["start"], x["end"]))
    merged = []
    i = 0
    while i < len(shaped):
        seg = shaped[i]
        if seg["duration"] < CONFIG.min_dur:
            merged_flag = False
            if i + 1 < len(shaped):
                comb = combine_segments(seg, shaped[i + 1])
                if comb["duration"] <= CONFIG.preferred_max_dur:
                    shaped[i + 1] = comb
                    merged_flag = True
            if not merged_flag and merged:
                comb = combine_segments(merged[-1], seg)
                if comb["duration"] <= CONFIG.preferred_max_dur:
                    merged[-1] = comb
                    merged_flag = True
            if not merged_flag:
                dropped_segments.append(
                    {
                        "row_id": row_id,
                        "segment_id": "",
                        "reason": "too_short_unmergeable",
                        "duration": seg["duration"],
                        "text": seg["text"],
                    }
                )
            i += 1
            continue
        merged.append(seg)
        i += 1

    valid = []
    for seg in merged:
        if seg["duration"] < CONFIG.min_dur:
            dropped_segments.append(
                {
                    "row_id": row_id,
                    "segment_id": "",
                    "reason": "below_min_duration",
                    "duration": seg["duration"],
                    "text": seg["text"],
                }
            )
            continue
        if seg["duration"] > CONFIG.max_dur:
            hard_parts = split_chunk_to_target(seg["words"], CONFIG.max_dur)
            for hp in hard_parts:
                hp_seg = make_segment(hp)
                if hp_seg["duration"] > CONFIG.max_dur:
                    dropped_segments.append(
                        {
                            "row_id": row_id,
                            "segment_id": "",
                            "reason": "above_hard_max_after_split",
                            "duration": hp_seg["duration"],
                            "text": hp_seg["text"],
                        }
                    )
                elif hp_seg["duration"] < CONFIG.min_dur:
                    dropped_segments.append(
                        {
                            "row_id": row_id,
                            "segment_id": "",
                            "reason": "below_min_after_hard_split",
                            "duration": hp_seg["duration"],
                            "text": hp_seg["text"],
                        }
                    )
                else:
                    valid.append(hp_seg)
            continue
        valid.append(seg)

    for seg_idx, seg in enumerate(valid):
        segment_candidates.append(
            {
                "row_id": row_id,
                "audio_path": rec["audio_path"],
                "file_duration": rec["duration"],
                "segment_index": seg_idx,
                "start": seg["start"],
                "end": seg["end"],
                "duration": seg["duration"],
                "text": seg["text"],
            }
        )

segment_candidates_df = pd.DataFrame(segment_candidates)
dropped_segments_df = pd.DataFrame(dropped_segments)
print("Segmentation candidates:", len(segment_candidates_df))
print("Dropped (segmentation stage):", len(dropped_segments_df))
print(segment_candidates_df.head(5))


# =============================================================================


In [ ]:
# ## Cell 11: Audio extraction + normalize + trim + metadata/logs
# =============================================================================
def trim_silence_with_cap(y, sr, max_trim_ms=200, db_threshold=-35.0):
    if y.size == 0:
        return y
    frame_length = max(256, int(sr * 0.02))
    hop_length = max(128, int(sr * 0.01))
    rms = librosa.feature.rms(
        y=y.astype(np.float32),
        frame_length=frame_length,
        hop_length=hop_length,
    )[0]
    if rms.size == 0:
        return y
    max_rms = float(np.max(rms))
    if max_rms <= 1e-8:
        return y

    threshold = max_rms * (10 ** (db_threshold / 20.0))
    active = np.where(rms >= threshold)[0]
    if active.size == 0:
        return y

    start_sample = int(librosa.frames_to_samples(int(active[0]), hop_length=hop_length))
    end_sample = int(librosa.frames_to_samples(int(active[-1]) + 1, hop_length=hop_length))
    start_sample = max(0, min(start_sample, len(y)))
    end_sample = max(start_sample, min(end_sample, len(y)))

    max_trim = int(sr * (max_trim_ms / 1000.0))
    left_trim = min(start_sample, max_trim)
    right_trim = min(len(y) - end_sample, max_trim)
    new_start = left_trim
    new_end = len(y) - right_trim
    if new_end <= new_start:
        return y
    return y[new_start:new_end]


def normalize_text_safe(text: str) -> str:
    return normalize_whitespace(text)


def ffmpeg_extract_to_wav16k_mono(src_path: str, start_sec: float, end_sec: float, out_path: Path):
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-ss",
        f"{start_sec:.3f}",
        "-to",
        f"{end_sec:.3f}",
        "-i",
        src_path,
        "-ac",
        "1",
        "-ar",
        str(CONFIG.target_sr),
        "-c:a",
        "pcm_s16le",
        str(out_path),
    ]
    subprocess.run(cmd, check=True)


metadata_rows = []
export_drop_rows = []

for rec in tqdm(segment_candidates, total=len(segment_candidates), desc="Exporting segments"):
    row_id = int(rec["row_id"])
    seg_idx = int(rec["segment_index"])
    audio_path = rec["audio_path"]
    text = str(rec["text"]).strip()
    if not text:
        export_drop_rows.append(
            {
                "row_id": row_id,
                "segment_id": f"{row_id:06d}_{seg_idx:03d}",
                "reason": "empty_text",
                "duration": rec["duration"],
                "text": text,
            }
        )
        continue

    pad = CONFIG.padding_ms / 1000.0
    start = max(0.0, float(rec["start"]) - pad)
    end = min(float(rec["file_duration"]), float(rec["end"]) + pad)
    if end <= start:
        export_drop_rows.append(
            {
                "row_id": row_id,
                "segment_id": f"{row_id:06d}_{seg_idx:03d}",
                "reason": "invalid_time_window",
                "duration": 0.0,
                "text": text,
            }
        )
        continue

    temp_wav = TEMP_DIR / f"tmp_{row_id:06d}_{seg_idx:03d}.wav"
    out_id = f"utt_{row_id:06d}_{seg_idx:03d}"
    out_wav = WAVS_DIR / f"{out_id}.wav"

    try:
        ffmpeg_extract_to_wav16k_mono(audio_path, start, end, temp_wav)
        y, sr = sf.read(temp_wav)
        if y.ndim > 1:
            y = np.mean(y, axis=1)
        y = y.astype(np.float32)
        peak = float(np.max(np.abs(y))) if y.size else 0.0
        if peak > 0.98 and peak > 0:
            y = y * (0.98 / peak)

        y = trim_silence_with_cap(
            y,
            sr=sr,
            max_trim_ms=CONFIG.silence_trim_ms,
            db_threshold=-35.0,
        )

        final_dur = float(len(y) / sr) if sr > 0 else 0.0
        if final_dur < CONFIG.min_dur:
            export_drop_rows.append(
                {
                    "row_id": row_id,
                    "segment_id": out_id,
                    "reason": "below_min_duration_after_trim",
                    "duration": final_dur,
                    "text": text,
                }
            )
            continue
        if final_dur > CONFIG.max_dur:
            export_drop_rows.append(
                {
                    "row_id": row_id,
                    "segment_id": out_id,
                    "reason": "above_max_duration_after_trim",
                    "duration": final_dur,
                    "text": text,
                }
            )
            continue

        sf.write(out_wav, y, sr, subtype="PCM_16")
        metadata_rows.append(
            {
                "filename": f"wavs/{out_wav.name}",
                "raw_text": text,
                "normalized_text": normalize_text_safe(text),
            }
        )
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        processing_errors.append(
            {
                "row_id": row_id,
                "audio_path": audio_path,
                "stage": "export",
                "error": msg,
                "traceback": traceback.format_exc(),
            }
        )
        export_drop_rows.append(
            {
                "row_id": row_id,
                "segment_id": out_id,
                "reason": f"export_error: {msg}",
                "duration": rec["duration"],
                "text": text,
            }
        )
    finally:
        if temp_wav.exists():
            try:
                temp_wav.unlink()
            except Exception:
                pass

metadata_df = pd.DataFrame(metadata_rows)
if metadata_df.empty:
    print("Warning: no segments exported.")

metadata_path = OUTPUT_DIR / "metadata.csv"
metadata_df.to_csv(
    metadata_path,
    sep="|",
    header=False,
    index=False,
    quoting=csv.QUOTE_MINIMAL,
)

all_drops_df = (
    pd.concat([dropped_segments_df, pd.DataFrame(export_drop_rows)], ignore_index=True)
    if len(dropped_segments_df) or len(export_drop_rows)
    else pd.DataFrame(columns=["row_id", "segment_id", "reason", "duration", "text"])
)
all_drops_path = OUTPUT_DIR / "dropped_segments.csv"
all_drops_df.to_csv(all_drops_path, index=False)

errors_df = pd.DataFrame(processing_errors)
errors_path = OUTPUT_DIR / "processing_errors.csv"
errors_df.to_csv(errors_path, index=False)

print("Export completed.")
print("- Exported wavs:", len(metadata_df))
print("- Metadata:", metadata_path)
print("- Dropped segments:", all_drops_path)
print("- Errors:", errors_path)
print(metadata_df.head(5))


# =============================================================================


In [ ]:
# ## Cell 12: Post-export QC + samples
# =============================================================================
exported_wavs = sorted(WAVS_DIR.glob("*.wav"))
post_qc_records = []

for wav_path in tqdm(exported_wavs, total=len(exported_wavs), desc="Post-QC"):
    y, sr = sf.read(wav_path, always_2d=True)
    n_samples, n_channels = y.shape
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    duration = float(n_samples / sr) if sr > 0 else 0.0
    post_qc_records.append(
        {
            "filename": f"wavs/{wav_path.name}",
            "path": str(wav_path),
            "duration_sec": duration,
            "sample_rate": int(sr),
            "channels": int(n_channels),
            "peak_amplitude": peak,
            "is_clipping": bool(peak >= 0.999),
        }
    )

qc_post_df = pd.DataFrame(post_qc_records)
qc_post_path = OUTPUT_DIR / "qc_post_export.csv"
qc_post_df.to_csv(qc_post_path, index=False)

if len(qc_post_df) > 0:
    print("Post-export QC summary:")
    print("- count:", len(qc_post_df))
    print(
        "- min/mean/median/max dur:",
        float(qc_post_df["duration_sec"].min()),
        float(qc_post_df["duration_sec"].mean()),
        float(qc_post_df["duration_sec"].median()),
        float(qc_post_df["duration_sec"].max()),
    )
    print("- % > 15s:", float((qc_post_df["duration_sec"] > 15).mean() * 100))
    print("- % clipping:", float((qc_post_df["peak_amplitude"] >= 0.999).mean() * 100))
else:
    print("No exported WAVs found for post-QC.")

print("Saved:", qc_post_path)

metadata_with_path = None
if (OUTPUT_DIR / "metadata.csv").exists():
    metadata_with_path = pd.read_csv(
        OUTPUT_DIR / "metadata.csv",
        sep="|",
        names=["filename", "raw_text", "normalized_text"],
        header=None,
    )

if metadata_with_path is not None and len(metadata_with_path) > 0:
    sample_n = min(5, len(metadata_with_path))
    samples = metadata_with_path.sample(sample_n, random_state=CONFIG.seed).copy()
    samples["abs_path"] = samples["filename"].apply(lambda x: str(OUTPUT_DIR / x))
    print("\nRandom samples:")
    for r in samples.itertuples(index=False):
        print("-", r.filename)
        print("  text:", r.raw_text)
        print("  path:", r.abs_path)
    # Optional inline playback in Colab/Jupyter:
    # from IPython.display import Audio, display
    # for r in samples.head(3).itertuples(index=False):
    #     display(Audio(filename=r.abs_path))
else:
    print("No metadata rows found; skipping samples.")


# =============================================================================


In [ ]:
# ## Cell 13: Zip final dataset + download helper
# =============================================================================
zip_base = str(OUTPUT_DIR)
zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(OUTPUT_DIR))
print("Final output directory:", OUTPUT_DIR)
print("Final zip path:", zip_path)
if Path(zip_path).exists():
    print("Zip size (MB):", round(Path(zip_path).stat().st_size / (1024 * 1024), 2))

try:
    from google.colab import files

    print("To download now, run:")
    print(f"files.download('{zip_path}')")
except Exception:
    print("Not in Colab environment (or google.colab unavailable).")


# =============================================================================


In [ ]:
# ## Cell 14 (Optional): HF Casablanca Algeria inspection (license warning)
# =============================================================================
# WARNING:
# The Casablanca dataset includes licensing terms (e.g., CC BY-NC-ND) that may restrict
# commercial use and derivative training workflows. Review the dataset card and license
# before any fine-tuning usage. This cell does NOT merge it with your dataset.

if CONFIG.run_optional_hf_cell:
    from datasets import load_dataset

    hf_ds = load_dataset("UBC-NLP/Casablanca", "Algeria")
    print(hf_ds)
    for split_name in hf_ds.keys():
        print(f"Split: {split_name}, rows: {len(hf_ds[split_name])}")
    print("\nFeatures:")
    print(hf_ds[list(hf_ds.keys())[0]].features)
    print("\nFirst 3 samples from first split:")
    first_split = list(hf_ds.keys())[0]
    for i in range(min(3, len(hf_ds[first_split]))):
        print(hf_ds[first_split][i])
else:
    print("Skipping optional HF Casablanca cell. Set CONFIG.run_optional_hf_cell=True to run it.")

# Optional export example (manual/isolated only, do not auto-merge):
# rows = []
# for item in hf_ds[first_split]:
#     text = item.get("text", "")
#     audio_path = item.get("audio", {}).get("path", "") if isinstance(item.get("audio"), dict) else ""
#     rows.append({"filename": audio_path, "raw_text": text, "normalized_text": text.strip()})
# pd.DataFrame(rows).to_csv("/content/casablanca_algeria_metadata.csv", sep="|", index=False, header=False)
